# DisasterM3 — QLoRA Fine-Tuning of Qwen2.5-VL-7B
**Purpose:** Reproduce the fine-tuning pipeline from Wang et al. (NeurIPS 2025).  
**Hardware:** Kaggle T4 GPU (16 GB VRAM) — QLoRA 4-bit to fit in memory.  
**Reference:** Appendix B.3 of arXiv:2505.21089v2.

### Deviations from Paper (logged in DEVIATIONS.md)
| Parameter | Paper (4×H100) | Ours (1×T4) |
|---|---|---|
| Precision | bf16 | fp16 (T4 lacks native bf16) |
| Quantization | None (full LoRA) | 4-bit NF4 (QLoRA) |
| Attention | FlashAttention-2 | SDPA (T4 is Turing cc7.5) |
| Per-device batch | 64 | 1–2 |
| Gradient accum. | 1 | 128–256 (to approximate global batch 256) |
| GPUs | 4 | 1 |

### What stays the same
LoRA rank=64, alpha=16, dropout=0.05, AdamW lr=2e-4, β₁=0.9, β₂=0.95,
cosine schedule, 1 epoch, vision encoder frozen.

In [ ]:
# ── Cell 1: Environment Setup ─────────────────────────────────────────────
# Pin exact versions to avoid Kaggle/torch conflicts.
# Run this cell FIRST, then restart the kernel before proceeding.
# IMPORTANT: Do NOT re-run this cell after restart — go straight to Cell 2.

!pip install -q \
    torch==2.4.1 \
    transformers==4.46.3 \
    peft==0.14.0 \
    bitsandbytes==0.45.0 \
    accelerate==1.2.1 \
    trl==0.13.0 \
    qwen-vl-utils==0.0.8 \
    pillow \
    huggingface_hub[hf_xet]

print("✓ Dependencies installed. RESTART THE KERNEL NOW, then skip to Cell 2.")

In [ ]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────
import os
import json
import torch
from pathlib import Path
from datetime import datetime

# ── Paths (adjust if data is mounted differently) ──
# Option A: Data is in a Kaggle Dataset mount
KAGGLE_INPUT = Path("/kaggle/input/disasterm3")
# Option B: Data is in /tmp (just downloaded)
TMP_DATA = Path("/tmp/disasterm3/DisasterM3/DisasterM3_Instruct")

# Auto-detect
if KAGGLE_INPUT.exists():
    DATA_ROOT = KAGGLE_INPUT
    print(f"✓ Using Kaggle Dataset mount: {DATA_ROOT}")
else:
    DATA_ROOT = TMP_DATA
    print(f"✓ Using /tmp data: {DATA_ROOT}")

MANIFEST_PATH = DATA_ROOT / "train_release.json"
IMAGE_DIR = Path("/tmp/train_images")  # Will unzip here
BOX_IMAGE_DIR = Path("/tmp/box_train_images")
MASK_DIR = Path("/tmp/masks")

# ── Model ──
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

# ── LoRA config (Appendix B.3) ──
LORA_RANK = 64
LORA_ALPHA = 16
LORA_DROPOUT = 0.05

# ── Training config ──
LEARNING_RATE = 2e-4
ADAM_BETA1 = 0.9
ADAM_BETA2 = 0.95
NUM_EPOCHS = 1
PER_DEVICE_BATCH = 1              # T4 constraint
GRADIENT_ACCUMULATION = 256       # Global batch ≈ 1 × 256 = 256
MAX_SEQ_LENGTH = 2048             # Qwen2.5-VL default
WARMUP_RATIO = 0.03
WEIGHT_DECAY = 0.01

# ── Checkpointing (12-hour Kaggle session cap) ──
CHECKPOINT_DIR = "/kaggle/working/checkpoints"
SAVE_STEPS = 50                   # Save every 50 optimizer steps
LOGGING_STEPS = 10

# ── Output ──
OUTPUT_DIR = "/kaggle/working/qwen25vl_disasterm3_qlora"
ADAPTER_NAME = f"disasterm3_qwen25vl7b_qlora_r{LORA_RANK}_a{LORA_ALPHA}"

print(f"✓ Config loaded")
print(f"  Model: {MODEL_ID}")
print(f"  LoRA: rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")
print(f"  Effective batch size: {PER_DEVICE_BATCH} × {GRADIENT_ACCUMULATION} = {PER_DEVICE_BATCH * GRADIENT_ACCUMULATION}")
print(f"  Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"  VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

## 1. Unzip Images
Only unzip to `/tmp` (ephemeral). The permanent copy stays in the Kaggle Dataset mount.

In [ ]:
# ── Cell 3: Unzip images ──────────────────────────────────────────────────
import zipfile
import time

def unzip_if_needed(zip_path, dest_dir, label):
    """Unzip only if destination doesn't already exist (resume-safe)."""
    if dest_dir.exists() and any(dest_dir.iterdir()):
        count = sum(1 for _ in dest_dir.rglob("*") if _.is_file())
        print(f"✓ {label}: already unpacked ({count:,} files)")
        return
    
    dest_dir.mkdir(parents=True, exist_ok=True)
    print(f"⏳ Unpacking {label}...")
    t0 = time.time()
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dest_dir.parent)
    elapsed = time.time() - t0
    count = sum(1 for _ in dest_dir.rglob("*") if _.is_file())
    print(f"✓ {label}: unpacked {count:,} files in {elapsed:.0f}s")

# Main training images (29.7 GB — this takes ~5-10 min)
train_zip = DATA_ROOT / "train_images.zip"
if train_zip.exists():
    unzip_if_needed(train_zip, IMAGE_DIR, "train_images")
else:
    print(f"⚠ {train_zip} not found — skipping main images")

# Box images for ORR/BBR tasks (2.96 GB)
box_zip = DATA_ROOT / "box_train_images.zip"
if box_zip.exists():
    unzip_if_needed(box_zip, BOX_IMAGE_DIR, "box_train_images")
else:
    print(f"⚠ {box_zip} not found — skipping box images")

# Masks for segmentation (173 MB)
mask_zip = DATA_ROOT / "masks.zip"
if mask_zip.exists():
    unzip_if_needed(mask_zip, MASK_DIR, "masks")
else:
    print(f"⚠ {mask_zip} not found — skipping masks")

## 2. Load & Prepare Training Data

In [ ]:
# ── Cell 4: Load manifest and build chat dataset ─────────────────────────
from PIL import Image

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    raw_data = json.load(f)

print(f"Loaded {len(raw_data):,} entries from manifest")

def resolve_image_path(rel_path):
    """Resolve a manifest-relative path to an absolute path on disk."""
    if not rel_path:
        return None
    # Normalize backslashes
    rel_path = rel_path.replace("\\", "/")
    # Try /tmp first (unzipped location)
    candidate = Path("/tmp") / rel_path
    if candidate.exists():
        return str(candidate)
    # Try Kaggle input
    candidate = DATA_ROOT / rel_path
    if candidate.exists():
        return str(candidate)
    return None

def entry_to_messages(entry):
    """Convert a manifest entry to Qwen2.5-VL chat messages format."""
    task = entry.get("task", "")
    
    # Build user content with images
    user_content = []
    
    if "image_path" in entry:
        # Relational reasoning: single post-disaster image with bounding boxes
        img_path = resolve_image_path(entry["image_path"])
        if img_path:
            user_content.append({"type": "image", "image": f"file://{img_path}"})
    else:
        # All other tasks: pre + post disaster pair
        pre_path = resolve_image_path(entry.get("pre_image_path", ""))
        post_path = resolve_image_path(entry.get("post_image_path", ""))
        if pre_path:
            user_content.append({"type": "image", "image": f"file://{pre_path}"})
        if post_path:
            user_content.append({"type": "image", "image": f"file://{post_path}"})
    
    # Skip entries where images couldn't be resolved
    if not any(c.get("type") == "image" for c in user_content):
        return None
    
    # Add prompt text
    prompt = entry.get("prompts", "")
    if isinstance(prompt, list):
        prompt = prompt[0]
    user_content.append({"type": "text", "text": prompt})
    
    # Get the training answer
    answer = entry.get("training_answer", "")
    if not answer:
        answer = entry.get("ground_truth", "")
    
    messages = [
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": [{"type": "text", "text": answer}]}
    ]
    return messages

# Convert all entries
print("Converting manifest entries to chat format...")
chat_dataset = []
skipped = 0
from collections import Counter
task_counts = Counter()

for entry in raw_data:
    msgs = entry_to_messages(entry)
    if msgs is not None:
        chat_dataset.append({
            "messages": msgs,
            "task": entry.get("task", "unknown"),
        })
        task_counts[entry.get("task", "unknown")] += 1
    else:
        skipped += 1

print(f"\n✓ Converted: {len(chat_dataset):,} entries")
print(f"  Skipped (missing images): {skipped:,}")
print(f"\nPer-task breakdown:")
for task, count in task_counts.most_common():
    print(f"  {task:<45} {count:>6,}")

## 3. Load Model with QLoRA

In [ ]:
# ── Cell 5: Load Qwen2.5-VL-7B with 4-bit quantization ───────────────────
from transformers import (
    Qwen2_5_VLForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# 4-bit quantization config (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # fp16 (T4 has no native bf16)
    bnb_4bit_use_double_quant=True,
)

print(f"⏳ Loading {MODEL_ID} in 4-bit...")
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",  # T4 can't use flash_attention_2
    torch_dtype=torch.float16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)

# Prepare for k-bit training
model = prepare_model_for_kbit_training(model)

print(f"✓ Model loaded")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"  Trainable before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

In [ ]:
# ── Cell 6: Apply LoRA adapters ───────────────────────────────────────────
# Target: LLM component only (vision encoder frozen) — matches paper's recipe.
# Find all linear layers in the language model to target.

lora_config = LoraConfig(
    r=LORA_RANK,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type="CAUSAL_LM",
    bias="none",
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✓ LoRA adapters applied")
print(f"  Trainable parameters: {trainable:,} ({trainable/total*100:.2f}%)")
print(f"  Total parameters:     {total:,}")
model.print_trainable_parameters()

## 4. Training Loop

In [ ]:
# ── Cell 7: Set up SFTTrainer ─────────────────────────────────────────────
from trl import SFTTrainer, SFTConfig
from transformers import TrainingArguments
import datasets

# Convert our chat_dataset list to a HuggingFace Dataset
hf_dataset = datasets.Dataset.from_list(chat_dataset)

# SFT training config
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=PER_DEVICE_BATCH,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    learning_rate=LEARNING_RATE,
    adam_beta1=ADAM_BETA1,
    adam_beta2=ADAM_BETA2,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    lr_scheduler_type="cosine",
    max_seq_length=MAX_SEQ_LENGTH,
    fp16=True,                         # T4: fp16 only
    bf16=False,                        # T4: no native bf16
    gradient_checkpointing=True,       # Save VRAM
    logging_steps=LOGGING_STEPS,
    save_steps=SAVE_STEPS,
    save_total_limit=3,                # Keep last 3 checkpoints
    report_to="none",                  # No wandb on Kaggle
    dataloader_num_workers=2,
    remove_unused_columns=False,
    max_grad_norm=1.0,                 # Gradient clipping (important for fp16)
    dataset_text_field=None,           # We use messages format
    dataset_kwargs={"skip_prepare_dataset": True},
)

print("✓ Training config ready")
print(f"  Steps per epoch: ~{len(chat_dataset) // (PER_DEVICE_BATCH * GRADIENT_ACCUMULATION)}")
print(f"  Total optimizer steps: ~{len(chat_dataset) // (PER_DEVICE_BATCH * GRADIENT_ACCUMULATION) * NUM_EPOCHS}")

In [ ]:
# ── Cell 8: Custom data collator for Qwen2.5-VL ──────────────────────────
from qwen_vl_utils import process_vision_info

def collate_fn(examples):
    """Custom collator that handles Qwen2.5-VL multi-image inputs."""
    texts = []
    image_inputs_list = []
    
    for example in examples:
        messages = example["messages"]
        
        # Apply Qwen2.5-VL chat template
        text = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
        
        # Process vision info (loads images from file:// URIs)
        image_inputs, video_inputs = process_vision_info(messages)
        image_inputs_list.append(image_inputs)
    
    # Tokenize with processor
    batch = processor(
        text=texts,
        images=image_inputs_list[0] if image_inputs_list else None,
        padding=True,
        truncation=True,
        max_length=MAX_SEQ_LENGTH,
        return_tensors="pt",
    )
    
    # Create labels (mask user tokens, only train on assistant response)
    labels = batch["input_ids"].clone()
    # Mask padding
    labels[labels == processor.tokenizer.pad_token_id] = -100
    batch["labels"] = labels
    
    return batch

print("✓ Custom collator defined")

In [ ]:
# ── Cell 9: Resume from checkpoint if available ─────────────────────────
resume_from = None
if os.path.exists(OUTPUT_DIR):
    checkpoints = [
        d for d in os.listdir(OUTPUT_DIR)
        if d.startswith("checkpoint-") and os.path.isdir(os.path.join(OUTPUT_DIR, d))
    ]
    if checkpoints:
        latest = max(checkpoints, key=lambda x: int(x.split("-")[1]))
        resume_from = os.path.join(OUTPUT_DIR, latest)
        print(f"✓ Resuming from checkpoint: {resume_from}")
    else:
        print("No existing checkpoints found — starting fresh")
else:
    print("No output directory found — starting fresh")

In [ ]:
# ── Cell 10: Train! ───────────────────────────────────────────────────────
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=hf_dataset,
    data_collator=collate_fn,
    processing_class=processor.tokenizer,
)

print(f"⏳ Starting training at {datetime.now().strftime('%H:%M:%S')}...")
print(f"   This will take several hours on a T4.")
print(f"   Checkpoints saved every {SAVE_STEPS} steps to {OUTPUT_DIR}")
print()

trainer.train(resume_from_checkpoint=resume_from)

print(f"\n✓ Training complete at {datetime.now().strftime('%H:%M:%S')}")

## 5. Save Adapter Checkpoint

In [ ]:
# ── Cell 11: Save the trained LoRA adapter ───────────────────────────────
adapter_path = os.path.join(OUTPUT_DIR, ADAPTER_NAME)
model.save_pretrained(adapter_path)
processor.save_pretrained(adapter_path)

print(f"✓ Adapter saved to: {adapter_path}")
print(f"  Contents:")
for f in sorted(os.listdir(adapter_path)):
    size = os.path.getsize(os.path.join(adapter_path, f)) / 1e6
    print(f"    {f:<40} {size:.1f} MB")

# Log training summary
summary = {
    "model": MODEL_ID,
    "adapter_name": ADAPTER_NAME,
    "lora_rank": LORA_RANK,
    "lora_alpha": LORA_ALPHA,
    "lora_dropout": LORA_DROPOUT,
    "learning_rate": LEARNING_RATE,
    "epochs": NUM_EPOCHS,
    "per_device_batch": PER_DEVICE_BATCH,
    "gradient_accumulation": GRADIENT_ACCUMULATION,
    "effective_batch_size": PER_DEVICE_BATCH * GRADIENT_ACCUMULATION,
    "precision": "fp16 + NF4 QLoRA",
    "attention": "sdpa",
    "training_entries": len(chat_dataset),
    "timestamp": datetime.now().isoformat(),
}
with open(os.path.join(adapter_path, "training_summary.json"), "w") as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Training summary written to {adapter_path}/training_summary.json")

## 6. Quick Sanity Check
Run a single inference to verify the adapter loaded correctly.

In [ ]:
# ── Cell 12: Sanity check inference ───────────────────────────────────────
# Pick a random training example and check the model can generate something
import random

test_entry = random.choice(chat_dataset)
test_messages = test_entry["messages"]

# Only keep the user message (remove assistant for generation)
user_msg = [test_messages[0]]

text = processor.apply_chat_template(
    user_msg,
    tokenize=False,
    add_generation_prompt=True,
)
image_inputs, _ = process_vision_info(user_msg)

inputs = processor(
    text=[text],
    images=image_inputs,
    return_tensors="pt",
).to(model.device)

with torch.no_grad():
    generated = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
    )

# Decode only the new tokens
output_text = processor.batch_decode(
    generated[:, inputs["input_ids"].shape[1]:],
    skip_special_tokens=True,
)[0]

print(f"Task: {test_entry['task']}")
print(f"Expected: {test_messages[1]['content'][0]['text'][:200]}...")
print(f"\nGenerated: {output_text[:200]}...")
print(f"\n✓ Sanity check passed — model generates text after fine-tuning.")